In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots

from pathlib import Path
from matplotlib.ticker import FuncFormatter

plt.style.use(["science", "ieee"])

base_font_old = 7.5
base_font_new = 10.5
legend_font_new = 8.0
axis_label_font = 13.5

scale_factor = base_font_new / base_font_old

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times"],
    "font.size": base_font_new,
    "axes.labelsize": axis_label_font,
    "xtick.labelsize": base_font_new,
    "ytick.labelsize": base_font_new,
    "legend.fontsize": legend_font_new,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

color_Ni = "#E69F00"
color_Fe = "#7B4AB5"
color_AC = "#D62728"

#Constants
R = 8.314462618
BAR_to_PA = 1e5
ATM_to_PA = 101325.0


# MODEL 1 — NICKEL

def reaction_rate_model_1(T, P_CH4_bar, P_H2_bar):

    P_CH4 = P_CH4_bar * BAR_to_PA
    P_H2 = P_H2_bar * BAR_to_PA

    A = 9.77054
    E = 88e3
    k = A * np.exp(-E / (R * T))

    A_CH4 = 6.887737e-10
    deltaH_CH4 = -56e3
    K_CH4 = A_CH4 * np.exp(-deltaH_CH4 / (R * T))

    A_H2 = 5.8009466e-16
    deltaH_H2 = -144e3
    K_H2 = A_H2 * np.exp(-deltaH_H2 / (R * T))

    Kp = 5.088e10 * np.exp(-91.2e3 / (R * T))

    numerator = k * (P_CH4 - P_H2**2 / Kp)

    denominator = (
        1
        + K_CH4 * P_CH4
        + K_H2 * P_H2**1.5
    )**2

    return numerator / denominator


# MODEL 2 — IRON

def reaction_rate_model_2(T, P_CH4_bar, P_H2_bar):

    P_CH4 = P_CH4_bar * BAR_to_PA / ATM_to_PA
    P_H2 = P_H2_bar * BAR_to_PA / ATM_to_PA

    P0 = 1.0 * BAR_to_PA / ATM_to_PA

    deltaG = (
        89658.88
        - 102.27 * T
        - 0.00428 * T**2
        - 2499358.99 / T
    )

    Keq = np.exp(
        -deltaG / (R * T)
    )

    Qp = P_H2**2 / (P_CH4 * P0)
    eta = Qp / Keq

    T_ref = 1073.0
    kp_ref = 1.3
    Ea = 160e3

    kp = kp_ref * np.exp(
        -(Ea / R)
        * (
            1.0 / T
            - 1.0 / T_ref
        )
    )

    return kp * P_CH4 * (1.0 - eta)


# MODEL 3 — ACTIVATED CARBON

def reaction_rate_model_3(T, P_CH4_bar):

    P_CH4 = P_CH4_bar * BAR_to_PA / ATM_to_PA

    n = 0.5
    kp0 = 1.4e10
    Ea = 216e3

    kp = kp0 * np.exp(
        -Ea / (R * T)
    )

    r_original = kp * P_CH4**n

    return r_original / 60.0

#Pressure baseline case
P_CH4_bar = 1.0
P_H2_bar = 0

#Temperature Ranges
T1_C = np.linspace(550, 700, 300)
T1_K = T1_C + 273.15

T2_C = np.linspace(700, 850, 300)
T2_K = T2_C + 273.15

T3_C = np.linspace(800, 950, 300)
T3_K = T3_C + 273.15


#Reaction rates
r1 = reaction_rate_model_1(
    T1_K,
    P_CH4_bar,
    P_H2_bar
)

r2 = reaction_rate_model_2(
    T2_K,
    P_CH4_bar,
    P_H2_bar
)

r3 = reaction_rate_model_3(
    T3_K,
    P_CH4_bar
)

r1_plot = np.where(
    r1 > 0,
    r1,
    np.nan
)

r2_plot = np.where(
    r2 > 0,
    r2,
    np.nan
)

r3_plot = np.where(
    r3 > 0,
    r3,
    np.nan
)

#Plot
fig, ax = plt.subplots(
    figsize=(
        3.6 * scale_factor,
        2.8 * scale_factor
    )
)

ax.plot(
    T1_C,
    r1_plot,
    color=color_Ni,
    linestyle="-",
    linewidth=1.2 * scale_factor,
    label=r"Nickel model $(550$--$700\,^\circ\mathrm{C})$"
)

ax.plot(
    T2_C,
    r2_plot,
    color=color_Fe,
    linestyle="-",
    linewidth=1.2 * scale_factor,
    label=r"Iron model $(700$--$850\,^\circ\mathrm{C})$"
)

ax.plot(
    T3_C,
    r3_plot,
    color=color_AC,
    linestyle="-",
    linewidth=1.2 * scale_factor,
    label=r"Carbon model $(800$--$950\,^\circ\mathrm{C})$"
)

ax.set_yscale("log")
ax.set_xlim(550, 950)

ax.set_xlabel(
    r"Temperature / [$^\circ\mathrm{C}$]",
    fontsize=axis_label_font
)

ax.set_ylabel(
    r"Reaction rate / "
    r"[$\mathrm{mol}_{\mathrm{CH_4}}\cdot"
    r"\mathrm{kg}_{\mathrm{cat}}^{\mbox{-}1}\cdot"
    r"\mathrm{s}^{\mbox{-}1}$]",
    fontsize=axis_label_font
)


def log_tick_formatter(value, pos=None):

    if value <= 0:
        return ""

    exponent = int(
        np.round(
            np.log10(value)
        )
    )

    # Only label exact powers of 10
    if not np.isclose(
        value,
        10.0**exponent
    ):
        return ""

    if exponent < 0:

        return (
            r"$10^{\mbox{-}"
            + str(abs(exponent))
            + r"}$"
        )

    return (
        r"$10^{"
        + str(exponent)
        + r"}$"
    )

ax.yaxis.set_major_formatter(
    FuncFormatter(
        log_tick_formatter
    )
)

ax.tick_params(
    axis="both",
    which="major",
    direction="in",
    top=True,
    right=True,
    labelsize=base_font_new,
    colors="black",
    length=3.5 * scale_factor,
    width=0.8 * scale_factor
)

ax.tick_params(
    axis="both",
    which="minor",
    direction="in",
    top=True,
    right=True,
    colors="black",
    length=2.0 * scale_factor,
    width=0.6 * scale_factor
)

for spine in ax.spines.values():

    spine.set_color(
        "black"
    )

    spine.set_linewidth(
        0.8 * scale_factor
    )

legend = ax.legend(
    fontsize=legend_font_new,
    loc="best",
    frameon=True,
    facecolor="white",
    edgecolor="black",
    handlelength=1.5,
    handletextpad=0.4,
    borderpad=0.35,
    labelspacing=0.30
)

for text in legend.get_texts():
    text.set_color("black")

fig.tight_layout(
    pad=0.8 * scale_factor
)

fig.savefig("reaction_rate_vs_temperature.pdf",
    format="pdf"
)

plt.show()

#Numerical Results
print()

print(
    f"Nickel Model | 550-700 °C | "
    f"min = {np.min(r1):.6e} | "
    f"max = {np.max(r1):.6e}"
)

print(
    f"Iron Model   | 700-850 °C | "
    f"min = {np.min(r2):.6e} | "
    f"max = {np.max(r2):.6e}"
)

print(
    f"Carbon Model | 800-950 °C | "
    f"min = {np.min(r3):.6e} | "
    f"max = {np.max(r3):.6e}"
)
